In [1]:
import os
import numpy as np
import pandas as pd
from torchinfo import summary
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset

In [2]:
num_classes = 72
lr = 1e-4
epochs = 20
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
dataset_dir = "/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated"

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness = 0.3, contrast = 0.3, saturation = 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])    # Std. ImageNet Normalization
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
def build_dataset(root, split, transform):
    datasets_list = []
    global_class_to_idx = {}
    current_idx = 0

    for crop in sorted(os.listdir(root)):
        split_path = os.path.join(root, crop, split)
        if not os.path.isdir(split_path):
            continue

        ds = datasets.ImageFolder(split_path, transform=transform)

        # Build local → global index mapping
        local_to_global = {}
        for local_name, local_idx in ds.class_to_idx.items():
            global_name = f"{crop}__{local_name}"
            if global_name not in global_class_to_idx:
                global_class_to_idx[global_name] = current_idx
                current_idx += 1
            local_to_global[local_idx] = global_class_to_idx[global_name]

        # Remap the actual sample labels
        ds.samples = [(path, local_to_global[label]) for path, label in ds.samples]
        ds.targets = [local_to_global[label] for label in ds.targets]

        datasets_list.append(ds)

    return ConcatDataset(datasets_list), global_class_to_idx


# Build
dataset_dir = "/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated"

train_ds, class_map = build_dataset(dataset_dir, "Train", transform_train)
val_ds,   _         = build_dataset(dataset_dir, "Val",   transform_val)
test_ds,  _         = build_dataset(dataset_dir, "Test",  transform_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(class_map)
print(f"Classes: {num_classes}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

print(dict(list(class_map.items())[:5]))

Classes: 29
Train: 53693 | Val: 12067 | Test: 1358
{'Apple__Apple Scab': 0, 'Apple__Black Rot': 1, 'Apple__Cedar Apple Rust': 2, 'Apple__Healthy': 3, 'Bell Pepper__Bacterial Spot': 4}


In [5]:
import torchvision.models as models
model = models.resnet50(weights="IMAGENET1K_V2")

In [6]:
from torch.optim import Adam

model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.fc.in_features, num_classes)
)
if torch.cuda.device_count() > 1:
    print(f"Using: {torch.cuda.device_count()} GPUS")
    model = nn.DataParallel(model)
model = model.to(device)

summary(model)

Using: 2 GPUS


Layer (type:depth-idx)                        Param #
DataParallel                                  --
├─ResNet: 1-1                                 --
│    └─Conv2d: 2-1                            9,408
│    └─BatchNorm2d: 2-2                       128
│    └─ReLU: 2-3                              --
│    └─MaxPool2d: 2-4                         --
│    └─Sequential: 2-5                        --
│    │    └─Bottleneck: 3-1                   75,008
│    │    └─Bottleneck: 3-2                   70,400
│    │    └─Bottleneck: 3-3                   70,400
│    └─Sequential: 2-6                        --
│    │    └─Bottleneck: 3-4                   379,392
│    │    └─Bottleneck: 3-5                   280,064
│    │    └─Bottleneck: 3-6                   280,064
│    │    └─Bottleneck: 3-7                   280,064
│    └─Sequential: 2-7                        --
│    │    └─Bottleneck: 3-8                   1,512,448
│    │    └─Bottleneck: 3-9                   1,117,184
│    │    └─Bo

In [7]:
best_val_acc = 0
adam = torch.optim.Adam(model.parameters(), lr=lr)
loss = nn.CrossEntropyLoss()

In [8]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)

    return total_loss / total, correct / total

In [10]:
for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion=loss, optimizer=adam)
    val_loss, val_accuracy     = run_epoch(model, val_loader, criterion=loss)

    print(f"Epoch: {epoch}\n"
         f"Train Loss: {train_loss:.4f}\tTrain Acc: {train_accuracy:.4f}\n"
         f"Val Loss: {val_loss:.4f}\tVal Acc: {val_accuracy:.4f}")

    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'class_map': class_map,
            'val_acc': val_accuracy
        }, '/kaggle/working/resnet50_phase1_best.pth')
        print(f"Saved best model (val_acc={val_accuracy:.4f})")

print(f"\nPhase 1 done. Best val acc: {best_val_acc:.4f}")

Epoch: 1
Train Loss: 0.0695	Train Acc: 0.9793
Val Loss: 0.0244	Val Acc: 0.9915
Saved best model (val_acc=0.9915)
Epoch: 2
Train Loss: 0.0291	Train Acc: 0.9914
Val Loss: 0.0118	Val Acc: 0.9969
Saved best model (val_acc=0.9969)
Epoch: 3
Train Loss: 0.0190	Train Acc: 0.9944
Val Loss: 0.0168	Val Acc: 0.9947
Epoch: 4
Train Loss: 0.0157	Train Acc: 0.9954
Val Loss: 0.0106	Val Acc: 0.9970
Saved best model (val_acc=0.9970)
Epoch: 5
Train Loss: 0.0131	Train Acc: 0.9960
Val Loss: 0.0206	Val Acc: 0.9933


KeyboardInterrupt: 